In [1]:
# 텍스트 셀 1: 드라이브 연결, 분할 불러오기
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
BASE = '/content/drive/MyDrive/emotion_ser'
sp = pd.read_csv(BASE + '/split_v1.csv', encoding='utf-8-sig')
print(sp['split'].value_counts())
print(sp[['발화문', 'y']].head())

Mounted at /content/drive
split
train    3500
test     1000
val       500
Name: count, dtype: int64
                                            발화문      y
0  일주일에 5일은 기본이고 이틀은 옵션이야 아휴 내가 기가 막힌다 기가 막혀.\t  angry
1                             벌써 한시간째야. 너무 힘들어.  angry
2                약속 장소에 도착한지 한시간이 넘었어! 정말 답답하다.  angry
3                        만날 때마다 약속 시간에 늦어서 속상해.  angry
4   친구가 약속시간에 너무 많이 늦어서 전화를 해봤어. 그런데 내 전화를 안받아.  angry


In [2]:
# 텍스트 셀 2: KcELECTRA 모델 준비
import torch, numpy as np
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from scipy.special import softmax

print('GPU:', torch.cuda.is_available())
MODEL = 'beomi/KcELECTRA-base'
labels = sorted(sp['y'].unique())
l2i = {l: i for i, l in enumerate(labels)}
tok = AutoTokenizer.from_pretrained(MODEL)

class TDS(Dataset):
    def __init__(self, df):
        self.enc = tok(list(df['발화문'].astype(str)), truncation=True, max_length=64)
        self.y = [l2i[v] for v in df['y']]
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}
        d['labels'] = self.y[i]
        return d

tr = sp[sp['split'] == 'train']
va = sp[sp['split'] == 'val']
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels), label2id=l2i, id2label={i: l for l, i in l2i.items()})

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pred, average='macro'), 'acc': accuracy_score(p.label_ids, pred)}

args = TrainingArguments(output_dir='/content/ckpt_t', per_device_train_batch_size=32, per_device_eval_batch_size=64, learning_rate=3e-5, warmup_steps=50, num_train_epochs=5, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, load_best_model_at_end=True, metric_for_best_model='macro_f1', fp16=True, logging_steps=50, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=TDS(tr), eval_dataset=TDS(va), data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)

GPU: True


config.json:   0%|          | 0.00/514 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
# 텍스트 셀 3: 학습
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Acc
1,1.289975,1.015429,0.590609,0.628000
2,0.800347,0.716278,0.755085,0.756000
3,0.485702,0.656944,0.776840,0.776000
4,0.381028,0.670269,0.777596,0.780000
5,0.280272,0.668968,0.781828,0.782000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=550, training_loss=0.6809042826565829, metrics={'train_runtime': 139.1708, 'train_samples_per_second': 125.745, 'train_steps_per_second': 3.952, 'total_flos': 240124083545232.0, 'train_loss': 0.6809042826565829, 'epoch': 5.0})

In [4]:
# 텍스트 셀 4: 시험 평가, 3분류 계산, 확률 저장
rows = []
for name in ['val', 'test']:
    s = sp[sp['split'] == name]
    pr = softmax(trainer.predict(TDS(s)).predictions, axis=1)
    r = s[['wav_id', 'y', 'split']].copy()
    for i, l in enumerate(labels):
        r['p_' + l] = pr[:, i]
    rows.append(r)
res = pd.concat(rows)
res.to_csv(BASE + '/text_probs_v1.csv', index=False, encoding='utf-8-sig')

t = res[res['split'] == 'test']
pred = np.array(labels)[t[['p_' + l for l in labels]].values.argmax(1)]
print(classification_report(t['y'], pred))
print('5분류 macro-F1:', f1_score(t['y'], pred, average='macro'))
print(labels)
print(confusion_matrix(t['y'], pred, labels=labels))

to3 = {'happy': 'good', 'neutral': 'flat', 'sad': 'bad', 'angry': 'bad', 'anxious': 'bad'}
y3 = t['y'].map(to3)
p3 = pd.Series(pred, index=t.index).map(to3)
print('3분류 macro-F1:', f1_score(y3, p3, average='macro'))

              precision    recall  f1-score   support

       angry       0.83      0.89      0.86       200
     anxious       0.88      0.95      0.91       200
       happy       0.84      0.83      0.83       200
     neutral       0.61      0.54      0.57       200
         sad       0.81      0.80      0.80       200

    accuracy                           0.80      1000
   macro avg       0.79      0.80      0.80      1000
weighted avg       0.79      0.80      0.80      1000

5분류 macro-F1: 0.7957772558394615
['angry', 'anxious', 'happy', 'neutral', 'sad']
[[177   1   1  14   7]
 [  0 191   0   5   4]
 [  2   1 166  28   3]
 [ 24  18  28 107  23]
 [ 10   7   3  21 159]]
3분류 macro-F1: 0.7703709965202267


In [5]:
# 융합 셀 1: 음성·텍스트 확률 파일 합치기
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np
BASE = '/content/drive/MyDrive/emotion_ser'
a = pd.read_csv(BASE + '/audio_probs_v2.csv', encoding='utf-8-sig')
t = pd.read_csv(BASE + '/text_probs_v1.csv', encoding='utf-8-sig')
labels = ['angry', 'anxious', 'happy', 'neutral', 'sad']
a = a.rename(columns={'p_' + l: 'a_' + l for l in labels})
t = t.rename(columns={'p_' + l: 't_' + l for l in labels})
m = a.merge(t, on=['wav_id', 'y', 'split'], how='inner')
print(len(a), len(t), len(m))
print(m['split'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1500 1500 1500
split
test    1000
val      500
Name: count, dtype: int64


In [6]:
# 융합 셀 2: 텍스트만 / 음성만 / 평균 / 학습형 융합 비교
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from sklearn.linear_model import LogisticRegression

A = ['a_' + l for l in labels]
T = ['t_' + l for l in labels]
to3 = {'happy': 'good', 'neutral': 'flat', 'sad': 'bad', 'angry': 'bad', 'anxious': 'bad'}
va = m[m['split'] == 'val']
te = m[m['split'] == 'test']
L = np.array(labels)

def score(y, pred):
    y3 = pd.Series(y).map(to3).values
    p3 = pd.Series(pred).map(to3).values
    nr = recall_score(y, pred, labels=['neutral'], average=None)[0]
    return f1_score(y, pred, average='macro'), f1_score(y3, p3, average='macro'), nr

def boot(y, pred, n=1000):
    rng = np.random.default_rng(0)
    y = np.asarray(y)
    pred = np.asarray(pred)
    v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        v.append(f1_score(y[i], pred[i], average='macro'))
    return np.percentile(v, [2.5, 97.5])

pa = te[A].values
pt = te[T].values
best_w, best_f = 0, -1
for w in np.arange(0, 1.01, 0.1):
    p = w * va[A].values + (1 - w) * va[T].values
    f = f1_score(va['y'], L[p.argmax(1)], average='macro')
    if f > best_f:
        best_w, best_f = w, f
print('검증으로 고른 음성 가중치:', round(best_w, 1))

X_va = np.hstack([np.log(va[A].values + 1e-6), np.log(va[T].values + 1e-6)])
X_te = np.hstack([np.log(te[A].values + 1e-6), np.log(te[T].values + 1e-6)])
lr = LogisticRegression(C=1.0, max_iter=2000)
lr.fit(X_va, va['y'])

preds = {
    '텍스트만': L[pt.argmax(1)],
    '음성만': L[pa.argmax(1)],
    '단순 평균(0.5)': L[(0.5 * pa + 0.5 * pt).argmax(1)],
    '가중 평균(검증으로 선택)': L[(best_w * pa + (1 - best_w) * pt).argmax(1)],
    '학습형 융합(로지스틱)': lr.predict(X_te),
}
rows = []
for k, p in preds.items():
    f5, f3, nr = score(te['y'].values, p)
    lo, hi = boot(te['y'].values, p)
    rows.append([k, round(f5, 3), f'{lo:.3f}~{hi:.3f}', round(f3, 3), round(nr, 2)])
print(pd.DataFrame(rows, columns=['방법', '5분류 F1', '5분류 95%구간', '3분류 F1', '중립 재현율']).to_string(index=False))
print(labels)
print(confusion_matrix(te['y'], preds['학습형 융합(로지스틱)'], labels=labels))

검증으로 고른 음성 가중치: 0.5
            방법  5분류 F1   5분류 95%구간  3분류 F1  중립 재현율
          텍스트만   0.796 0.773~0.818   0.770    0.54
           음성만   0.658 0.628~0.686   0.796    0.72
    단순 평균(0.5)   0.842 0.819~0.863   0.827    0.70
가중 평균(검증으로 선택)   0.842 0.819~0.863   0.827    0.70
  학습형 융합(로지스틱)   0.872 0.850~0.890   0.863    0.74
['angry', 'anxious', 'happy', 'neutral', 'sad']
[[185   1   1   7   6]
 [  0 193   0   3   4]
 [  2   1 172  22   3]
 [ 16   8  17 149  10]
 [  9   5   2  10 174]]


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, subprocess
import numpy as np, pandas as pd, torch
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

BASE = '/content/drive/MyDrive/emotion_ser'
MP = BASE + '/model_w2v2_v2'
labels = ['angry', 'anxious', 'happy', 'neutral', 'sad']
to3 = {'happy': 'good', 'neutral': 'flat', 'sad': 'bad', 'angry': 'bad', 'anxious': 'bad'}
SR = 16000

fe = Wav2Vec2FeatureExtractor.from_pretrained(MP)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MP, torch_dtype=torch.float32).eval()
print('모델 라벨 순서:', model.config.id2label)

def load(path):
    out = subprocess.run(['ffmpeg', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', str(SR), '-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(out, dtype=np.float32)

def predict(path):
    a = load(path)
    ch = [a[i:i + SR * 6] for i in range(0, len(a), SR * 6)]
    if len(ch) > 1 and len(ch[-1]) < SR:
        ch = ch[:-1]
    ps = []
    with torch.no_grad():
        for c in ch:
            x = fe(c, sampling_rate=SR, return_tensors='pt')
            ps.append(torch.softmax(model(**x).logits, -1)[0].numpy())
    return np.mean(ps, 0)

files = sorted(glob.glob(BASE + '/my_test/*'))
print('파일 개수:', len(files))
rows = []
for f in files:
    name = os.path.splitext(os.path.basename(f))[0]
    p = predict(f)
    top5 = labels[int(p.argmax())]
    exp = name.split('_')[-1] if name.split('_')[-1] in ('good', 'flat', 'bad') else '?'
    pred3 = to3[top5]
    rows.append([name, exp, top5, pred3, '정답' if exp == pred3 else ('-' if exp == '?' else '오답')]
                + [round(float(x), 2) for x in p])
df = pd.DataFrame(rows, columns=['파일', '의도(3분류)', '예측(5분류)', '예측(3분류)', '판정'] + labels)
print(df.to_string(index=False))
ok = df[df['의도(3분류)'] != '?']
print('3분류 정답률:', (ok['판정'] == '정답').sum(), '/', len(ok))

KeyboardInterrupt: 